# Euribor Rate Explorer

**Data source:** [euriborrates.com](https://euriborrates.com/) — an independent, non-commercial, non-profit information resource.

This notebook fetches Euribor fixings (1W, 1M, 3M, 6M, 12M) and displays the current state of our dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

from pricebook.data.euribor_loader import (
    fetch_date, fetch_year_all_tenors, load_local, save_to_csv,
    summary, attribution, TENORS, TENOR_LABELS,
)
from pricebook.viz import configure_theme
from pricebook.viz._theme import get_theme
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod
from pricebook.curves.bootstrap import bootstrap
from pricebook.core.day_count import DayCountConvention
import matplotlib.pyplot as plt

configure_theme()
theme = get_theme()
print(attribution())

## 1. Fetch Today's Fixing

In [ ]:
# Fetch today's rates (all 5 tenors)
today = date.today()
fixing = fetch_date(today)

if fixing:
    print(f"Euribor fixings for {fixing.date}")
    print(f"{'Tenor':<12} {'Rate':>8}")
    print("-" * 22)
    for tenor in TENORS:
        rate = fixing.rates.get(tenor)
        if rate is not None:
            print(f"{TENOR_LABELS[tenor]:<12} {rate*100:>7.3f}%")
else:
    print(f"No fixing available for {today} (weekend or holiday?)")

## 2. Fetch a Sample Year (all 5 tenors)

In [ ]:
# Fetch 2024 with all 5 tenors (5 requests, ~10s)
fixings_2024 = fetch_year_all_tenors(2024, delay_between_tenors=1.5)
print(f"Fetched {len(fixings_2024)} business days for 2024")

# Convert to DataFrame
rows = []
for f in fixings_2024:
    row = {"date": f.date}
    for t in TENORS:
        row[t] = f.rates.get(t)
    rows.append(row)

df_2024 = pd.DataFrame(rows).set_index("date")
df_2024.columns = [TENOR_LABELS[t] for t in TENORS]
df_2024 = df_2024 * 100  # convert to percentage

print(f"\nFirst 5 days:")
df_2024.head()

In [ ]:
# Last 5 days
df_2024.tail()

## 3. Term Structure Snapshot

Current Euribor curve shape: how rates vary by maturity.

In [ ]:
latest = df_2024.iloc[-1]
tenor_months = [1/4, 1, 3, 6, 12]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tenor_months, latest.values, "o-", color=theme.colors[0], 
        linewidth=theme.line_width, markersize=8)
ax.set_xlabel("Tenor (months)")
ax.set_ylabel("Rate (%)")
ax.set_title(f"Euribor Term Structure — {df_2024.index[-1]}\nSource: euriborrates.com",
             fontsize=theme.title_size)
ax.set_xticks(tenor_months)
ax.set_xticklabels(["1W", "1M", "3M", "6M", "12M"])
ax.grid(True, alpha=theme.grid_alpha, color=theme.grid_color)
plt.tight_layout()
plt.show()

## 4. Rate History (2024)

All 5 tenors through the year — observe the ECB easing cycle.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for i, col in enumerate(df_2024.columns):
    ax.plot(df_2024.index, df_2024[col], label=col, 
            color=theme.colors[i % len(theme.colors)], linewidth=theme.line_width)

ax.set_xlabel("Date")
ax.set_ylabel("Rate (%)")
ax.set_title("Euribor Rates — 2024\nSource: euriborrates.com", fontsize=theme.title_size)
ax.legend(loc="upper right")
ax.grid(True, alpha=theme.grid_alpha, color=theme.grid_color)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Curve Slope (12M − 1W spread)

Positive = normal (upward sloping). Negative = inverted.

In [ ]:
slope = df_2024["12 Months"] - df_2024["1 Week"]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.fill_between(slope.index, slope.values, 0, 
                where=slope >= 0, color=theme.colors[2], alpha=0.4, label="Normal")
ax.fill_between(slope.index, slope.values, 0, 
                where=slope < 0, color=theme.colors[1], alpha=0.4, label="Inverted")
ax.plot(slope.index, slope.values, color=theme.foreground, linewidth=0.8)
ax.axhline(0, color=theme.foreground, linewidth=0.5)
ax.set_ylabel("Spread (pp)")
ax.set_title("Euribor Slope: 12M − 1W (2024)\nSource: euriborrates.com",
             fontsize=theme.title_size)
ax.legend()
ax.grid(True, alpha=theme.grid_alpha, color=theme.grid_color)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Build EUR Discount Curve from Euribor Fixings

Use pricebook's `bootstrap()` to build a real `DiscountCurve` from today's Euribor deposits.

In [ ]:
# Build DiscountCurve from latest Euribor fixings
# Map tenors to maturity dates
latest_fixing = fixings_2024[-1]
ref_date = latest_fixing.date

tenor_offsets = {
    "1w": relativedelta(weeks=1),
    "1m": relativedelta(months=1),
    "3m": relativedelta(months=3),
    "6m": relativedelta(months=6),
    "12m": relativedelta(months=12),
}

deposits = []
for tenor in TENORS:
    rate = latest_fixing.rates.get(tenor)
    if rate is not None:
        mat = ref_date + tenor_offsets[tenor]
        deposits.append((mat, rate))

# Bootstrap the curve (EUR convention: ACT/360 for deposits)
curve = bootstrap(
    ref_date, deposits, swaps=[],
    deposit_day_count=DayCountConvention.ACT_360,
    interpolation=InterpolationMethod.LOG_LINEAR,
)

print(f"EUR Discount Curve — {ref_date}")
print(f"{'Tenor':<8} {'Maturity':<12} {'DF':>10} {'Zero Rate':>10}")
print("-" * 44)
for tenor, (mat, rate) in zip(TENORS, deposits):
    df = curve.df(mat)
    zr = curve.zero_rate(mat) * 100
    print(f"{TENOR_LABELS[tenor]:<8} {mat} {df:>10.6f} {zr:>9.3f}%")

print(f"\nSource: euriborrates.com")

In [ ]:
# Plot discount factors and zero rates from the bootstrapped curve
plot_dates = [ref_date + timedelta(days=d) for d in range(1, 366)]
dfs = [curve.df(d) for d in plot_dates]
zeros = [curve.zero_rate(d) * 100 for d in plot_dates]
days = [(d - ref_date).days for d in plot_dates]

# Pillar positions
pillar_days = [(ref_date + tenor_offsets[t] - ref_date).days for t in TENORS]
pillar_dfs = [curve.df(ref_date + tenor_offsets[t]) for t in TENORS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(days, dfs, color=theme.colors[0], linewidth=theme.line_width)
ax1.scatter(pillar_days, pillar_dfs, color=theme.colors[1], zorder=5, s=50, label="Euribor pillars")
ax1.set_xlabel("Days")
ax1.set_ylabel("Discount Factor")
ax1.set_title(f"EUR Discount Curve — {ref_date}", fontsize=theme.title_size)
ax1.legend()
ax1.grid(True, alpha=theme.grid_alpha, color=theme.grid_color)

ax2.plot(days, zeros, color=theme.colors[2], linewidth=theme.line_width)
ax2.set_xlabel("Days")
ax2.set_ylabel("Zero Rate (%)")
ax2.set_title(f"EUR Zero Rate Curve — {ref_date}", fontsize=theme.title_size)
ax2.grid(True, alpha=theme.grid_alpha, color=theme.grid_color)

fig.suptitle("Source: euriborrates.com", fontsize=9, y=0.02, color="gray")
plt.tight_layout()
plt.show()

## 7. Statistics

In [ ]:
print("2024 Euribor Statistics (in %)")
print("=" * 55)
stats = df_2024.describe().T[["mean", "std", "min", "max"]]
stats.columns = ["Mean", "StdDev", "Min", "Max"]
print(stats.round(3).to_string())
print(f"\nBusiness days: {len(df_2024)}")
print(f"Date range: {df_2024.index[0]} to {df_2024.index[-1]}")
print(f"\nData source: euriborrates.com")

---

## Next Steps

1. **Full historical load** — fetch 1999–2026, all 5 tenors (~135 requests)
2. **Historical curve database** — bootstrap a `DiscountCurve` for every business day, store NS/Svensson parameters
3. **Daily cron** — automated daily update via `update_latest()`
4. **Analytics** — PCA on curve moves, regime detection, ECB impact, carry/roll-down backtest

*Data sourced from [euriborrates.com](https://euriborrates.com/) — an independent, non-commercial, non-profit information resource.*